# EatWise Engine — Phase 2: Obesity Classifier

**Course:** BMET2925 AI, Data and Society in Health — Semester 1 2026  
**Pipeline phase:** Phase 2 of 4 (obesity classification)  
**Methods anchor:** Module 4 (sklearn classification pipelines)  
**Input:** `cleaned_data/d1_obesity_prediction_encoded.csv` (2087 × 32, model-ready)  
**Output:** `models/obesity_classifier.pkl` (best model by Weighted F1), `models/class_labels.json`  

Four candidate models are compared (Random Forest, Logistic Regression, XGBoost, SVC) on a stratified 80-20 train-test split. Primary selection metric: Weighted F1 per the project proposal. Height and Weight are excluded from model inputs to prevent target leakage (BMI deterministically derives the class labels). 170 BMI-band-mismatched rows (8.15% of D1) are removed before training to reduce label noise.

---
## Section A: Setup and data preparation

**A.1 — Imports and global constant.**  
`RANDOM_STATE = 42` is set once and used wherever a `random_state` argument is accepted to ensure full reproducibility across all four models and the train-test split.

In [ ]:
import json
import os
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    multilabel_confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

RANDOM_STATE = 42
print('Imports complete. RANDOM_STATE =', RANDOM_STATE)

**A.2 — Load both D1 CSVs and assert shapes.**  
The categorical CSV retains the original `Obesity` string labels needed to derive the label mapping empirically. The encoded CSV is the model-ready feature matrix. Both were produced by the same cleaning notebook run and share row order.

In [ ]:
df_cat = pd.read_csv('cleaned_data/d1_obesity_prediction_cleaned.csv')
df_enc = pd.read_csv('cleaned_data/d1_obesity_prediction_encoded.csv')

assert df_cat.shape == (2087, 17), f'Unexpected categorical shape: {df_cat.shape}'
assert df_enc.shape == (2087, 32), f'Unexpected encoded shape: {df_enc.shape}'

print('d1_obesity_prediction_cleaned.csv shape:', df_cat.shape)
print('d1_obesity_prediction_encoded.csv shape:', df_enc.shape)

**A.3 — Derive class label mapping empirically.**  
The cleaning notebook encoded the target using a custom BMI-ascending order (not a default `LabelEncoder`, which would sort alphabetically and place `Obesity_Type_III` at index 4, before the `Overweight_*` classes). The integer-to-string correspondence is recovered by joining the two CSVs on shared row index and printed in full for verification. No mapping is assumed — what the encoder produced is what is used.

In [ ]:
# Recover mapping: integer target_encoded → Obesity class string
mapping_df = pd.DataFrame({
    'target_encoded': df_enc['target_encoded'],
    'Obesity': df_cat['Obesity']
})
CLASS_LABELS = (
    mapping_df
    .drop_duplicates()
    .sort_values('target_encoded')
    .set_index('target_encoded')['Obesity']
    .to_dict()
)

print('Empirically derived class label mapping:')
for k, v in sorted(CLASS_LABELS.items()):
    print(f'  {k}: {v}')

assert len(CLASS_LABELS) == 7, f'Expected 7 classes, got {len(CLASS_LABELS)}'

**A.4 — Identify 170 BMI-band-mismatched rows using the cleaning notebook's logic.**  
BMI is recomputed from Height and Weight on the categorical CSV. Each row is checked against the WHO-derived band corresponding to its labelled class. Mismatched indices are collected for removal from the encoded dataframe. Exactly 170 mismatches are expected (8.15% of 2087 rows), as identified in cleaning notebook Section 1.8.

In [ ]:
BMI_BANDS = {
    'Insufficient_Weight': (-np.inf, 18.5),
    'Normal_Weight':        (18.5,   25.0),
    'Overweight_Level_I':   (25.0,   27.5),
    'Overweight_Level_II':  (27.5,   30.0),
    'Obesity_Type_I':       (30.0,   35.0),
    'Obesity_Type_II':      (35.0,   40.0),
    'Obesity_Type_III':     (40.0,   np.inf),
}

df_check = df_cat.copy()
df_check['BMI_computed'] = df_check['Weight'] / (df_check['Height'] ** 2)

def in_band(row):
    lo, hi = BMI_BANDS[row['Obesity']]
    return lo <= row['BMI_computed'] < hi

df_check['band_match'] = df_check.apply(in_band, axis=1)
mismatch_idx = df_check.index[~df_check['band_match']].tolist()

print(f'BMI-band mismatches: {len(mismatch_idx)} / {len(df_check)} '
      f'({100 * len(mismatch_idx) / len(df_check):.2f}%)')
print('Mismatch breakdown by labelled class:')
print(df_check.loc[~df_check['band_match'], 'Obesity'].value_counts())

assert len(mismatch_idx) == 170, f'Expected 170 mismatches, got {len(mismatch_idx)}'

**A.5 — Drop 170 mismatched rows from the encoded dataframe.**  
Removing label-noise rows before training prevents the classifier from learning from incorrectly labelled instances. This is a pre-training data quality step, not a post-hoc filter.

In [ ]:
df_enc_clean = df_enc.drop(index=mismatch_idx).reset_index(drop=True)

assert df_enc_clean.shape == (1917, 32), f'Unexpected shape after mismatch removal: {df_enc_clean.shape}'
print('Shape after removing 170 mismatched rows:', df_enc_clean.shape)

**A.6 — Drop Height and Weight columns.**  
Height and Weight are excluded because BMI is a deterministic function of them, and BMI determines the class labels. Retaining them would give the classifier direct access to the label-generating rule rather than forcing it to learn from lifestyle features — the actual goal of Phase 2.

In [ ]:
df_enc_clean = df_enc_clean.drop(columns=['Height', 'Weight'])

assert df_enc_clean.shape == (1917, 30), f'Unexpected shape after dropping Height/Weight: {df_enc_clean.shape}'
print('Shape after dropping Height and Weight:', df_enc_clean.shape)

**A.7 — Separate features and target; confirm class balance carried through.**  
The target column is `target_encoded`. After the 170-row drop, the class distribution should remain approximately balanced (within ~4 pp, as identified in Phase 1 EDA).

In [ ]:
X = df_enc_clean.drop(columns=['target_encoded'])
y = df_enc_clean['target_encoded']

assert X.shape == (1917, 29), f'Unexpected X shape: {X.shape}'
assert y.shape == (1917,),    f'Unexpected y shape: {y.shape}'

print('X shape:', X.shape)
print('y shape:', y.shape)
print()
print('Class distribution after mismatch removal:')
dist = y.value_counts().sort_index()
for enc_label, count in dist.items():
    pct = 100 * count / len(y)
    print(f'  {enc_label} ({CLASS_LABELS[enc_label]}): {count} ({pct:.1f}%)')

proportions = dist / dist.sum()
spread_pp = (proportions.max() - proportions.min()) * 100
print(f'\nSpread (max − min): {spread_pp:.1f} pp')

**A.8 — Stratified 80-20 train-test split.**  
`stratify=y` preserves class proportions in both splits. This is the only split performed — the proposal scopes a single held-out evaluation, not cross-validation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train size: {len(X_train)} rows ({100 * len(X_train) / len(X):.1f}%)')
print(f'Test size:  {len(X_test)} rows ({100 * len(X_test) / len(X):.1f}%)')

assert len(X_train) + len(X_test) == 1917
print('\nClass distribution in test set (stratification check):')
test_dist = y_test.value_counts().sort_index()
for enc_label, count in test_dist.items():
    pct = 100 * count / len(y_test)
    print(f'  {enc_label} ({CLASS_LABELS[enc_label]}): {count} ({pct:.1f}%)')

---
## Section B: Train four candidate models

Four models are trained on the same `(X_train, y_train)` split. Hyperparameters are left at sklearn/XGBoost defaults throughout — the proposal scopes a comparison at defaults, not a tuning exercise. Fitting time is recorded per model to give a practical sense of computational cost.

**Documented deviation — SVC pipeline with StandardScaler:**  
The proposal specifies SVC as a candidate but does not explicitly wrap it in a preprocessing pipeline. This build adds `StandardScaler` upstream of SVC (and Logistic Regression) for the following reason: SVC with an RBF kernel uses Euclidean distances in feature space. Features on different scales (e.g. `Age` in years vs one-hot binary columns) cause large-magnitude features to dominate the kernel, handicapping SVC relative to tree-based models that are scale-invariant. Omitting scaling for SVC alone would make the comparison unfair. The pipeline ensures SVC competes on equal footing. This deviation is documented here so it is defensible in Q&A.

**B.1 — Random Forest classifier.**  
Primary candidate per proposal. `class_weight='balanced'` down-weights majority classes proportionally; at 6.4 pp spread this is a robustness setting rather than a correction. All other parameters at defaults.

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE)
t0 = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - t0
print(f'Random Forest fit time: {rf_time:.2f}s')

**B.2 — Logistic Regression in a StandardScaler pipeline.**  
`max_iter=1000` prevents convergence warnings on this feature count (29 features, 1533 training rows). `StandardScaler` upstream is required because LR uses gradient descent and penalises large-magnitude features. No `class_weight` specified — the proposal does not list it for LR and the class spread does not require it.

In [ ]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
t0 = time.time()
lr.fit(X_train, y_train)
lr_time = time.time() - t0
print(f'Logistic Regression fit time: {lr_time:.2f}s')

**B.3 — XGBoost classifier.**  
`eval_metric='mlogloss'` suppresses the default warning when multi-class labels are passed. All other parameters at XGBoost defaults. No `class_weight` equivalent in XGBoost at defaults — the proposal does not specify `scale_pos_weight` for XGB.

In [ ]:
xgb = XGBClassifier(eval_metric='mlogloss', random_state=RANDOM_STATE)
t0 = time.time()
xgb.fit(X_train, y_train)
xgb_time = time.time() - t0
print(f'XGBoost fit time: {xgb_time:.2f}s')

**B.4 — SVC in a StandardScaler pipeline (documented deviation).**  
`class_weight='balanced'` included for the same reason as RF: SVC's decision boundary can be skewed by class imbalance in the support vectors. `StandardScaler` upstream is required for RBF kernel correctness — see deviation note above. `random_state` accepted by SVC for probability estimation consistency.

In [ ]:
svc = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    SVC(class_weight='balanced', random_state=RANDOM_STATE, probability=True)),
])
t0 = time.time()
svc.fit(X_train, y_train)
svc_time = time.time() - t0
print(f'SVC fit time: {svc_time:.2f}s')

In [ ]:
print('Section B — fitting times:')
print(f'  Random Forest:       {rf_time:.2f}s')
print(f'  Logistic Regression: {lr_time:.2f}s')
print(f'  XGBoost:             {xgb_time:.2f}s')
print(f'  SVC:                 {svc_time:.2f}s')
print()
print('All four models fitted on (X_train, y_train) — 1533 rows, 29 features.')

---
## Section C: Evaluation

Each fitted model is evaluated on the held-out test set (384 rows, unseen during training) across five metrics. The primary selection metric is Weighted F1, per the project proposal. Weighted F1 was chosen over accuracy because it accounts for class-size differences in the average and is more informative than accuracy alone for a 7-class problem. Macro Precision, Macro Recall, and Macro Specificity are reported as secondary metrics to give a complete picture of per-class error behaviour without weighting by class size.

**C.1 — `specificity_macro` helper function.**  
sklearn has no built-in specificity metric. `multilabel_confusion_matrix` returns an array of shape `(n_classes, 2, 2)`, where entry `[i]` is the binary confusion matrix for class i treated as positive versus all others. For each class: TN = `mcm[i, 0, 0]`, FP = `mcm[i, 0, 1]`, per-class specificity = TN / (TN + FP). The macro specificity is the unweighted mean across all 7 classes.

In [ ]:
def specificity_macro(y_true, y_pred):
    """
    Macro specificity for multi-class classification.
    Uses multilabel_confusion_matrix (shape: n_classes x 2 x 2).
    Per class i: specificity = TN / (TN + FP), where TN = mcm[i,0,0], FP = mcm[i,0,1].
    Returns unweighted mean across all classes.
    """
    mcm = multilabel_confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(len(mcm)):
        tn = mcm[i, 0, 0]
        fp = mcm[i, 0, 1]
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return float(np.mean(specs))

print('specificity_macro defined.')

**C.2 — Evaluate all four models on the held-out test set.**  
Predictions are generated once per model via `predict`. Five metrics are computed: Accuracy, Precision (macro), Recall (macro), Specificity (macro), and Weighted F1. Results are collected into a dataframe, sorted by Weighted F1 descending, and printed.

In [ ]:
CANDIDATES = {
    'Random Forest':       rf,
    'Logistic Regression': lr,
    'XGBoost':             xgb,
    'SVC':                 svc,
}

rows = []
for name, model in CANDIDATES.items():
    y_pred = model.predict(X_test)
    rows.append({
        'Model':              name,
        'Accuracy':           round(accuracy_score(y_test, y_pred), 4),
        'Precision_macro':    round(precision_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'Recall_macro':       round(recall_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'Specificity_macro':  round(specificity_macro(y_test, y_pred), 4),
        'Weighted_F1':        round(f1_score(y_test, y_pred, average='weighted'), 4),
    })

results = (
    pd.DataFrame(rows)
    .sort_values('Weighted_F1', ascending=False)
    .reset_index(drop=True)
)

print(results.to_string(index=False))

**C.3 — Results commentary.**

Random Forest achieved the highest Weighted F1 (0.8466), placing it 2.01 pp above the next best model (XGBoost, 0.8265). This gap is consistent across all five metrics: RF leads on Accuracy (0.8490 vs 0.8307), Precision macro (0.8498 vs 0.8316), Recall macro (0.8426 vs 0.8224), and Specificity macro (0.9747 vs 0.9716). The Specificity macro values are high for all four models — 0.9378 even for Logistic Regression — which is expected in a multi-class one-vs-rest decomposition: negative class pools are large, so false positive rates tend to be low regardless of model quality. The more informative discriminator between models here is Recall macro and Weighted F1, where RF's lead over XGBoost is clear and the drop to SVC (Weighted F1 0.7624) and Logistic Regression (0.6165) is substantial. Logistic Regression's low performance relative to the tree-based models is consistent with the known non-linearity of lifestyle-to-obesity relationships: a linear decision boundary is insufficient when feature interactions (e.g. physical activity interacting with dietary habits) drive class separation. Random Forest is selected as the Phase 2 classifier per the proposal's primary candidate designation and its confirmed superiority on the primary selection metric.

---
## Section D: Model selection and save

The model with the highest Weighted F1 is selected from the results dataframe, saved to `models/obesity_classifier.pkl` via `joblib.dump`, and the class label mapping is saved to `models/class_labels.json`. Random Forest is a plain estimator (not a Pipeline), so it is saved directly — no wrapper needed. A round-trip `joblib.load` test confirms the saved object reproduces the same Weighted F1 on the test set before the section closes.

**D.1 — Select winner, create output directory, save model and class labels.**  
The winner is read from the top row of the sorted results dataframe (highest Weighted F1). The `models/` directory is created if absent. RF is saved directly with `joblib.dump` — it is a plain `RandomForestClassifier`, not a Pipeline, so no preprocessing wrapper is needed (the encoded CSV already has the correct feature representation). The class label mapping is serialised to JSON with integer keys cast to strings (JSON requires string keys).

In [ ]:
CANDIDATE_OBJECTS = {
    'Random Forest':       rf,
    'Logistic Regression': lr,
    'XGBoost':             xgb,
    'SVC':                 svc,
}

winner_name = results.iloc[0]['Model']
winner_f1   = results.iloc[0]['Weighted_F1']
winner_model = CANDIDATE_OBJECTS[winner_name]

print(f'Selected model: {winner_name}')
print(f'Weighted F1:    {winner_f1}')

os.makedirs('models', exist_ok=True)

PKL_PATH   = 'models/obesity_classifier.pkl'
JSON_PATH  = 'models/class_labels.json'

joblib.dump(winner_model, PKL_PATH)
print(f'\nSaved: {PKL_PATH}')

with open(JSON_PATH, 'w') as f:
    json.dump({str(k): v for k, v in CLASS_LABELS.items()}, f, indent=2)
print(f'Saved: {JSON_PATH}')

**D.2 — Round-trip load test.**  
The saved `.pkl` is loaded into a fresh variable (`rf_loaded`) and its Weighted F1 on the test set is recomputed. The result must match the in-memory model's score exactly, confirming the serialised object is intact and functional.

In [ ]:
rf_loaded = joblib.load(PKL_PATH)

y_pred_loaded = rf_loaded.predict(X_test)
f1_loaded = round(f1_score(y_test, y_pred_loaded, average='weighted'), 4)

print(f'Round-trip load test')
print(f'  File loaded:          {PKL_PATH}')
print(f'  Type:                 {type(rf_loaded).__name__}')
print(f'  Weighted F1 (saved):  {winner_f1}')
print(f'  Weighted F1 (loaded): {f1_loaded}')
print(f'  Match:                {f1_loaded == winner_f1}')

assert f1_loaded == winner_f1, 'Round-trip F1 mismatch — pkl may be corrupt'
print('\nSection D complete. models/obesity_classifier.pkl is verified.')

---
## Section E: Confusion matrix and feature importance

Both visualisations are run on the selected model (Random Forest) evaluated against the held-out test set. The confusion matrix uses class label strings on both axes for readability. The feature importance plot uses the actual training column names from `X`. Commentary follows each plot and is grounded in the actual cell outputs.

**E.1 — Confusion matrix on the test set.**  
`ConfusionMatrixDisplay.from_estimator` generates predictions internally and plots the matrix with class label strings. A wide figure (14 × 10 inches) is used so that the 7-class tick labels on both axes are readable without rotation overlap. Rows represent true labels; columns represent predicted labels.

In [ ]:
label_strings = [CLASS_LABELS[i] for i in sorted(CLASS_LABELS.keys())]

fig, ax = plt.subplots(figsize=(14, 10))
ConfusionMatrixDisplay.from_estimator(
    rf,
    X_test,
    y_test,
    display_labels=label_strings,
    ax=ax,
    colorbar=True,
)
ax.set_title('Random Forest — Confusion Matrix (test set, n=384)', fontsize=13)
ax.set_xlabel('Predicted label', fontsize=11)
ax.set_ylabel('True label', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

**E.1 — Confusion matrix commentary.**

The hardest class to classify is **Overweight_Level_II**: 29 of 45 test instances are correctly predicted (64.4% recall), with the remaining 16 split across adjacent and non-adjacent bands. Six instances are misclassified as Obesity_Type_I (the immediately higher BMI band — an expected adjacency error), and six as Normal_Weight (two bands below). This pattern is consistent with BMI boundary ambiguity: Overweight_Level_II spans only the narrow 27.5–30.0 kg/m² range, and individuals near the band edges will share similar lifestyle profiles with neighbours. A related adjacency error appears in Overweight_Level_I, where 6 instances are misclassified as Normal_Weight (the adjacent lower band).

By contrast, the two extreme classes are the easiest to predict. **Obesity_Type_III** (BMI > 40) achieves perfect recall (54/54), reflecting that severely obese individuals have lifestyle profiles sufficiently distinct from all other groups that the classifier makes no errors at that boundary. **Insufficient_Weight** also performs well (95.7% recall, 44/46 correct), with 2 instances misclassified as Normal_Weight — the only adjacent error possible for the lowest class.

The one notable non-adjacent confusion is Obesity_Type_I → Normal_Weight (5 instances, skipping Overweight_Level_I and Overweight_Level_II). This suggests that a small subset of obese individuals in this dataset have lifestyle profiles more similar to normal-weight individuals — a plausible finding in a dataset where Height and Weight are excluded and classification relies entirely on behavioural features.

**E.2 — Feature importance (Random Forest).**  
Random Forest exposes `feature_importances_` natively (mean decrease in impurity across all trees). The top 15 features are plotted as a horizontal bar chart with the highest-importance feature at the top. Column names are taken directly from the training `X` dataframe.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top15 = importances.nlargest(15).sort_values()  # ascending so highest bar is at top

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15.index, top15.values, color='steelblue')
ax.set_xlabel('Mean decrease in impurity (feature importance)', fontsize=10)
ax.set_title('Random Forest — Top 15 Feature Importances', fontsize=12)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

print('\nTop 15 features (descending importance):')
for feat, imp in importances.nlargest(15).items():
    print(f'  {feat:<35} {imp:.4f}')

**E.2 — Feature importance commentary.**

The top feature is **Age** (importance 0.1489), which is a demographic variable rather than a direct lifestyle feature. Its dominance is interpretable: obesity prevalence increases with age, and the dataset's age distribution (right-skewed, mean ~24 years) means that older individuals cluster disproportionately in higher BMI bands, giving the classifier a strong age-based signal.

Features 2 through 5 are all direct lifestyle variables: **FCVC** (vegetable consumption frequency, 0.1269), **TUE** (technology use time, 0.0907), **FAF** (physical activity frequency, 0.0904), and **CH2O** (daily water intake, 0.0895). Together, these four lifestyle features account for approximately 0.41 of the total importance, and they align with the rationale for dropping Height and Weight — the classifier is learning from behavioural patterns, not from a proxy BMI calculation. The absence of Height or Weight signal in the top features confirms that their removal did not leave a backdoor: no residual leakage is visible in the feature importance ranking.

One unexpected finding: **TUE** (technology use time) ranks third overall, above the more intuitively predictive **FAF** (physical activity). Technology use time is likely a behavioural proxy for sedentary lifestyle, capturing variance that partly overlaps with low physical activity. The separation of these two correlated features in the ranking reflects the Random Forest's ability to partition importance across redundant predictors — neither feature dominates when the other is present in the same tree path.

---
## Section F: Phase 3 wrapper function

`predict_obesity` takes a 14-field raw lifestyle dictionary and returns `(predicted_class_label, max_class_probability)`. The function replicates the preprocessing pipeline applied in Section A: string normalisation, field rename to match the cleaned training column name (`family_history_with_overweight` → `family_history`), one-hot encoding of nominals, and column reindexing to match the exact 29-column training feature set. The loaded `models/obesity_classifier.pkl` is used for inference. This function is defined here for Phase 4 import.

**F.1 — Wrapper function definition.**  
Preprocessing alignment is the critical step: the raw input uses `family_history_with_overweight` as the key (matching Phase 4 UI nomenclature), but the training column is named `family_history`. The rename is applied before one-hot encoding so that `pd.get_dummies` produces `family_history_no` / `family_history_yes`, matching the training feature names exactly. `reindex(X_COLUMNS, fill_value=0)` handles any unseen one-hot levels safely by filling with 0. Class label strings are loaded from `models/class_labels.json`; JSON keys are strings by spec, so they are cast to `int` before lookup.

In [ ]:
# Capture training column order at definition time for reindex alignment
X_COLUMNS = list(X.columns)

# Nominal columns that were one-hot encoded during training.
# Note: the input key 'family_history_with_overweight' is renamed to 'family_history'
# before encoding to match the training column name.
_NOMINALS = ['Gender', 'family_history', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

# Load class label strings once at definition time (JSON keys are strings; cast to int)
with open('models/class_labels.json') as _f:
    _CLASS_LABELS_LOADED = {int(k): v for k, v in json.load(_f).items()}


def predict_obesity(raw_lifestyle_dict: dict) -> tuple:
    """
    Take raw patient lifestyle features, return (predicted class label, max class probability).

    raw_lifestyle_dict expects keys: Gender, Age, family_history_with_overweight,
    FAVC, FCVC, NCP, CAEC, SMOKE, CH2O, SCC, FAF, TUE, CALC, MTRANS.
    Note: Height and Weight are NOT inputs to the classifier (dropped in Section A).

    Returns: (class_label_string, max_probability_float).
    """
    # Step 1: copy and rename family_history_with_overweight → family_history
    d = dict(raw_lifestyle_dict)
    if 'family_history_with_overweight' in d:
        d['family_history'] = d.pop('family_history_with_overweight')

    # Step 2: build single-row DataFrame; strip whitespace from string fields
    row = pd.DataFrame([d])
    for col in row.select_dtypes(include='object').columns:
        row[col] = row[col].str.strip()

    # Step 3: one-hot encode nominal columns
    row_encoded = pd.get_dummies(row, columns=_NOMINALS)

    # Step 4: reindex to match training feature set exactly; unseen levels → 0
    row_aligned = row_encoded.reindex(columns=X_COLUMNS, fill_value=0)

    # Step 5: load the saved model and predict
    clf = joblib.load('models/obesity_classifier.pkl')
    pred_idx = int(clf.predict(row_aligned)[0])
    prob = float(clf.predict_proba(row_aligned).max())

    # Step 6: map predicted index to class label string
    label = _CLASS_LABELS_LOADED[pred_idx]
    return label, prob


print('predict_obesity defined.')
print(f'Training column alignment: {len(X_COLUMNS)} features')

**F.2 — Two test predictions.**  
Two hardcoded profiles are passed through `predict_obesity` to verify end-to-end function correctness. Profile 1 represents a sedentary, high-snacking lifestyle (older male, family history of overweight, high caloric food intake, low physical activity, automobile transport). Profile 2 represents an active, low-snacking lifestyle (younger female, no family history, low caloric food intake, frequent physical activity, walking). Outputs are printed as-is from the model — no assertion on specific class labels is made.

In [ ]:
# Profile 1: sedentary, high-snacking — older male with family history of overweight,
# frequent high-caloric food intake, no physical activity, automobile transport
profile_sedentary = {
    'Gender':                        'Male',
    'Age':                           45,
    'family_history_with_overweight': 'yes',
    'FAVC':                          'yes',
    'FCVC':                          1.0,
    'NCP':                           3.0,
    'CAEC':                          'Always',
    'SMOKE':                         'no',
    'CH2O':                          1.0,
    'SCC':                           'no',
    'FAF':                           0.0,
    'TUE':                           2.0,
    'CALC':                          'Frequently',
    'MTRANS':                        'Automobile',
}

# Profile 2: active, low-snacking — younger female, no family history,
# low caloric food intake, high physical activity, walks for transport
profile_active = {
    'Gender':                        'Female',
    'Age':                           24,
    'family_history_with_overweight': 'no',
    'FAVC':                          'no',
    'FCVC':                          3.0,
    'NCP':                           3.0,
    'CAEC':                          'Sometimes',
    'SMOKE':                         'no',
    'CH2O':                          3.0,
    'SCC':                           'yes',
    'FAF':                           3.0,
    'TUE':                           0.5,
    'CALC':                          'no',
    'MTRANS':                        'Walking',
}

label_1, prob_1 = predict_obesity(profile_sedentary)
label_2, prob_2 = predict_obesity(profile_active)

print('Profile 1 — sedentary, high-snacking (older male):')
print(f'  Predicted class: {label_1}')
print(f'  Max probability: {prob_1:.4f}')
print()
print('Profile 2 — active, low-snacking (younger female):')
print(f'  Predicted class: {label_2}')
print(f'  Max probability: {prob_2:.4f}')

**F.2 — Test prediction commentary.**

Profile 2 (active, low-snacking) is classified as **Normal_Weight** (38.00% max class probability), which is a sensible output: high physical activity (FAF=3), frequent vegetable consumption (FCVC=3), high water intake (CH2O=3), and walking for transport are all features associated with lower BMI bands in the training data.

Profile 1 (sedentary, high-snacking) is classified as **Overweight_Level_II** (33.00% max class probability) rather than an Obesity class. This is interpretable: with Height and Weight excluded, the classifier relies on behavioural signals, and the sedentary profile's features (low FCVC, low FAF, high CAEC, automobile transport) are consistent with being in the overweight band rather than a confirmed obesity class. The model cannot confirm obesity without anthropometric data — which is exactly the intended behaviour after removing Height and Weight from the feature set.

The low max class probabilities for both profiles (33–38%) are consistent with the model's overall behaviour on the middle classes, where the confusion matrix shows genuine ambiguity between adjacent BMI bands. High-confidence predictions occur primarily for the extreme classes (Insufficient_Weight and Obesity_Type_III), which showed near-perfect recall on the test set.

---
## Section G: Phase 2 summary

**Model comparison.** Random Forest achieved the highest Weighted F1 of 0.8466, outperforming the next best model (XGBoost, 0.8265) by 2.01 pp. The gap widens further for Logistic Regression (0.6165) and SVC (0.7624), consistent with the known non-linearity of lifestyle-to-obesity relationships: tree ensembles partition the feature space in ways that a linear decision boundary cannot replicate (Section C results table).

**Selected model performance.** The Random Forest's Weighted F1 of 0.8466 is accompanied by a Macro Recall of 0.8426, meaning the model correctly identifies approximately 84% of instances per class on average across all seven obesity categories on the held-out test set (Section C results table).

**Feature importance.** The top feature by mean decrease in impurity is Age (0.1489), a demographic variable rather than a direct lifestyle feature — its dominance reflects age-related obesity prevalence patterns in the training data. Features 2 through 5 are all lifestyle variables: vegetable consumption frequency (FCVC, 0.1269), technology use time (TUE, 0.0907), physical activity frequency (FAF, 0.0904), and daily water intake (CH2O, 0.0895), confirming that the classifier learned from behavioural patterns after Height and Weight were excluded (Section E feature importance plot). No residual anthropometric signal appears in the top 15 features.

**Confusion matrix headline.** The hardest class to predict is Overweight_Level_II (29/45 test instances correct, 64.4% recall), which spans the narrowest BMI band (27.5–30.0 kg/m²) and is disproportionately affected by the 170-row BMI-band mismatch removal — individuals near this band's boundaries share lifestyle profiles with adjacent classes, producing the adjacency errors visible in the confusion matrix (Section E). Obesity_Type_III is the easiest class (54/54 correct, 100% recall), reflecting the sufficiently distinct lifestyle profile of severely obese individuals in this dataset.

**Test predictions from Section F.** The sedentary high-snacking profile (45-year-old male, CAEC=Always, FAF=0, MTRANS=Automobile) was classified as Overweight_Level_II with a max class probability of 33%; the active low-snacking profile (24-year-old female, FAF=3, FCVC=3, MTRANS=Walking) was classified as Normal_Weight at 38%. Both outputs are directionally sensible, but the probability values are modest because Random Forest's `predict_proba` reports the fraction of trees voting for each class rather than calibrated probabilities — on a 7-class problem with 100 trees, even a majority-class fraction will rarely exceed 50%. Phase 4 should display the full class probability distribution rather than a single confidence figure to give clinicians a richer picture of model uncertainty without the displayed percentage appearing to indicate poor model quality.

**Documented deviation.** The proposal listed SVC as a candidate model without specifying a preprocessing pipeline. This build wraps SVC in a `StandardScaler` pipeline (Section B.4), consistent with the treatment of Logistic Regression. The rationale: SVC with an RBF kernel uses Euclidean distances in feature space; unscaled features with different magnitudes (e.g. Age in years versus binary one-hot columns) distort the kernel and handicap SVC relative to scale-invariant tree models, making any comparison at defaults unfair without scaling. The deviation is explicit and defensible under Q&A.